# One protocol for every experiment: baseline family (B0, D1, A1, T1, DT1), YOLO26 experiments (Y1-Y6) and training-subset runs (SR, SC), selected by `EXPERIMENT`

This notebook is the experiment protocol. It declares the paths, takes the run-time decisions, calls the `auric` package, and shows the evidence (tables, plots, samples) behind each decision. All reusable logic lives in `src/auric`: label parsing, split, tiling, class-aware sampling, NMS/stitching, training, metrics. The notebook re-implements none of it.

| Stage | What happens | Written to `work_dir` |
|---|---|---|
| 0 Setup | runtime, code copy (Kaggle), config, every path declared once | |
| 1 Data audit | distributions that later decisions rest on: class balance, density, object size and shape, label quality | `audit/` |
| 2 Train/dev split | image-level dev split carved out of train, then checked against train_core and val | (split file ships with the code) |
| 3 Input formulation | **tiles**: build or reuse the training tile cache, tile statistics, read-back check. **scenes** (Y2): no cache; the scale at which each whole scene enters the network | `tiles/<key>/`, `audit/` |
| 4 YOLO dataset | run directory, train/dev lists (tiles or scenes), `data.yaml`; the class balance the detector will be trained on (class-aware sampler and class-weight preview) | `runs/<run_id>/data/` |
| 5 Training | model from `model.weights`, fixed schedule, full record of the run, what the imbalance levers did | `runs/<run_id>/` |
| 6 Evaluation | full-image inference on dev and val, metrics and error analysis | `runs/<run_id>/predictions/`, `eval/` |

**Experiments** (`configs/<EXPERIMENT>.yaml`; each states only what it changes from its `_base_`)

| `EXPERIMENT` | Model | Input | What it tests |
|---|---|---|---|
| `baseline` (B0), `D1`, `A1`, `T1` | YOLO11s | tiles | the earlier runs: updates per epoch (D1), 2x up-sampling (A1), single-class detector (T1) |
| `Y1` | YOLO26s | tiles | control of the Y series; Y1 vs D1 isolates the architecture |
| `Y2` | YOLO26s | whole scenes, `imgsz` 3008 | does the full-scene formulation of xView work better or worse than tiles |
| `Y3` | YOLO26s | tiles | `cls_pw` 0.5: inverse-frequency class weights in the classification loss |
| `Y4` | YOLO26s | tiles | classification-loss gain `cls` 0.5 -> 2.0 |
| `Y5` | YOLO26s | tiles | class-aware sampler (`train.sampler`, alpha 0) |
| `Y6` | YOLO26s | tiles | sampler + class weights together |
| `SR`, `SC` | YOLO11s | tiles of a **subset** of the train_core images | assignment 5.4: a random (`SR`) or a coverage-selected (`SC`) subset with an instance budget `subset.budget_frac`; run name, budget and subset seed are overrides, see the config files |

How the detections are decoded and merged (one-to-many head vs one-to-one head, class-aware vs class-agnostic NMS) is an *inference* question that needs no retraining: it is answered on the trained weights in `02_inference_variants.ipynb`.

**What goes where**
* `configs/<EXPERIMENT>.yaml` defines *what is computed*: split, input formulation, tiling, model, training, inference and evaluation parameters.
* `configs/env/<runtime>.yaml` defines *where it runs*: `data_root`, `work_dir`, device, workers.
* Decisions taken when running (what to do with an existing run, smoke mode, which samples to show) are explicit variables in the setup cells, not config values.

**Integrity rules.** Val (22 images) is never used to choose anything; every choice uses `dev`. No xView-pretrained weights are used, because val scenes very probably come from xView.


## Stage 0 · Setup

On Kaggle the package is uploaded as a dataset. Inputs are read-only, so it is copied to `/kaggle/working/src` and imported from there. The dataset path is set in `configs/env/kaggle.yaml`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

RUNTIME = "kaggle"     # "kaggle" | "local": selects configs/env/<RUNTIME>.yaml and where the code is imported from
EXPERIMENT = "D1"   # configs/<EXPERIMENT>.yaml: "baseline" | "D1" | "A1" | "T1" | "DT1" | "SR" | "SC" (YOLO11s)  or  "Y1" ... "Y6" (YOLO26s, see the table above)

if RUNTIME == "kaggle":
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4 per run; with two visible GPUs Ultralytics halves the dataloader workers
    SRC_INPUT = Path("/kaggle/input/datasets/shriramprajapati/src-auric/src/auric")    # <-- the Kaggle dataset that holds this submission's src/auric folder (read-only)
    SRC_DIR = Path("/kaggle/working/src")                                     # working copy; imported from here
    shutil.copytree(SRC_INPUT, SRC_DIR / "auric", dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.179"], check=True)
elif RUNTIME == "local":
    SRC_DIR = Path("..").resolve() / "src"      # the submission's src/ (run the notebook from notebooks/)
else:
    raise ValueError(f"unknown RUNTIME {RUNTIME!r}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

CONFIG_FILE = SRC_DIR / "auric" / "configs" / f"{EXPERIMENT}.yaml"       # experiment definition
ENV_FILE = SRC_DIR / "auric" / "configs" / "env" / f"{RUNTIME}.yaml"     # data_root, work_dir, device, workers

In [ ]:
# Run-time decisions: explicit choices for this execution, deliberately not part of the experiment config
SMOKE = False             # True: run "smoke" = 1 epoch on 10% of the training tiles (checks wiring and timing)
IF_RUN_EXISTS = "error"   # existing run directory: "error" | "resume" (continue, or reuse a finished run) | "overwrite"
VERBOSE_TRAIN = False     # False: tqdm progress bars; True: Ultralytics' own log
SAMPLE_SEED = 0           # selects the samples shown in visual inspections

OVERRIDES = []            # dotted config overrides, e.g. ["train.args.batch=16"]; they end up in the run's config.yaml
if SMOKE:
    OVERRIDES += ["experiment.name=smoke", "train.args.epochs=1", "train.args.fraction=0.1", "train.args.close_mosaic=0"]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

from auric.config import formulation, load_config
from auric.data.dataset import index_split, load_annotations, read_class_names, read_image
from auric.data.sampling import class_aware_probs, class_loss_weights, describe_sampling
from auric.data.splits import ensure_split, subset
from auric.data.subsets import describe_subset, select_subset
from auric.data.stats import image_table, object_table
from auric.data.tiling import TilingParams, count_uncovered, ensure_tile_cache, read_tile, verify_tile_labels
from auric.evaluation import evaluate
from auric.model.predict import TiledPredictor, stitch_ground_truth
from auric.model.train import fetch_pretrained, train_yolo, write_yolo_dataset, write_yolo_scene_dataset
from auric.runs import create_run, log_stage, open_run, read_record, run_id, sha256
from auric.structures import save_detections
from auric.visualize import CLASS_COLORS, crop_around, downscale, draw_boxes

pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

cfg = load_config(CONFIG_FILE, ENV_FILE, overrides=OVERRIDES)
SCENES = formulation(cfg) == "scenes"     # True: whole scenes are the training samples (no tile cache)

# every location used below, declared once from the two configured roots
DATA_ROOT = Path(cfg["paths"]["data_root"])
WORK_DIR = Path(cfg["paths"]["work_dir"])
SPLIT_FILE = Path(cfg["paths"]["split_file"])
TILING = TilingParams(**cfg["tiling"])
TILE_DIR = WORK_DIR / "tiles" / TILING.key
AUDIT_DIR = WORK_DIR / "audit"
PRETRAINED_DIR = WORK_DIR / "pretrained"
RUN_DIR = WORK_DIR / "runs" / run_id(cfg)
DEVICE, WORKERS = cfg["runtime"]["device"], cfg["runtime"]["workers"]

display(pd.DataFrame({"path": {k: str(v) for k, v in [
    ("config", CONFIG_FILE), ("env", ENV_FILE), ("data_root", DATA_ROOT), ("work_dir", WORK_DIR),
    ("split_file", SPLIT_FILE), ("tile_cache", TILE_DIR), ("audit", AUDIT_DIR), ("pretrained", PRETRAINED_DIR),
    ("run", RUN_DIR)]}}))
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"torch {torch.__version__} | GPU: {gpu} | device={DEVICE} workers={WORKERS} | overrides: {OVERRIDES or 'none'}")
print(f"{cfg['experiment']['name']}: model {cfg['model']['weights']} | input: {'whole scenes' if SCENES else 'tiles'} | "
      f"sampler: {cfg['train'].get('sampler') or 'plain shuffle'} | cls_pw {cfg['train']['args'].get('cls_pw', 0.0)} | "
      f"cls gain {cfg['train']['args']['cls']} | inference head {cfg['inference'].get('head', 'model default')}")

In [ ]:
# display helpers (notebook-only: matplotlib layout around auric.visualize)
def rgb(path):
    return np.ascontiguousarray(read_image(path)[:, :, ::-1])


def show(img, title="", width=14):
    plt.figure(figsize=(width, width * img.shape[0] / img.shape[1]))
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()


def crop_grid(items, title, cols=8, px=112, context=1.5):
    # items: (image_path, box_xyxy, rgb_colour, caption); crops are upscaled with nearest-neighbour for display
    if not items:
        print(f"{title}: none")
        return
    images = {p: rgb(p) for p in {str(t[0]) for t in items}}
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.9, rows * 2.1), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (path, box, colour, caption) in zip(axes.flat, items):
        crop, (x0, y0) = crop_around(images[str(path)], box, context)
        crop = draw_boxes(crop, [np.subtract(box, [x0, y0, x0, y0])], [colour], 1)
        ax.imshow(crop, interpolation="nearest")
        ax.set_title(caption, fontsize=7)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## Stage 1 · Data audit

Questions the later stages depend on:
1. **Inventory and density.** How many images and objects, and are train and val the same regime?
2. **Class balance.** How imbalanced are the classes, and how concentrated is each class in a few images? This sets how noisy per-class AP is and how the dev split must be stratified.
3. **Object size and shape.** Size per class and per subset, and aspect ratios. These set the tile size, resolution, overlap and size buckets.
4. **Label quality.** Degenerate, duplicated or conflicting boxes.

Everything is computed from the parsed labels (`auric.data.stats`). The tables are saved to `work_dir/audit/`.

In [ ]:
NAMES = read_class_names(DATA_ROOT)
K = len(NAMES)
SHORT = [n.replace("Truck w/", "").replace("Truck ", "").replace(" Truck", "") for n in NAMES]
COLORS = [np.array(c) / 255 for c in CLASS_COLORS]
RECORDS = {s: index_split(DATA_ROOT, s) for s in ("train", "val")}
ANNS = {s: load_annotations(RECORDS[s], K) for s in RECORDS}
PATHS = {(s, r.image_id): r.image_path for s in RECORDS for r in RECORDS[s]}

size_bins = cfg["evaluation"]["size_bins"]
objects = pd.concat([object_table(RECORDS[s], ANNS[s], NAMES, size_bins).assign(subset=s) for s in RECORDS],
                    ignore_index=True)
objects["short"] = pd.Categorical(np.asarray(SHORT)[objects.cls], categories=SHORT)
images = pd.concat([image_table(RECORDS[s], ANNS[s], NAMES).assign(subset=s) for s in RECORDS], ignore_index=True)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
objects.to_csv(AUDIT_DIR / "objects.csv", index=False)
images.to_csv(AUDIT_DIR / "images.csv", index=False)

inventory = images.groupby("subset").agg(
    images=("image_id", "size"), objects=("n_objects", "sum"), empty_images=("n_objects", lambda x: int((x == 0).sum())),
    obj_per_img_median=("n_objects", "median"), obj_per_img_mean=("n_objects", "mean"), obj_per_img_max=("n_objects", "max"),
    width_min=("width", "min"), width_max=("width", "max"), height_min=("height", "min"), height_max=("height", "max"))
inventory.to_csv(AUDIT_DIR / "inventory.csv")
display(inventory)

### 1.1 Class balance and concentration
"top-k share" is the fraction of a class's instances found in its k richest images. A high value means fewer independent examples than the raw count suggests.

In [ ]:
rows = []
for s in RECORDS:
    sub = images[images.subset == s]
    for n, short in zip(NAMES, SHORT):
        c = sub[f"n {n}"]
        rows.append({"subset": s, "class": short, "instances": int(c.sum()), "share %": 100 * c.sum() / sub.n_objects.sum(),
                     "images": int((c > 0).sum()), "inst / image (present)": c[c > 0].mean(),
                     "top-1 image share %": 100 * c.max() / c.sum(), "top-10 images share %": 100 * c.nlargest(10).sum() / c.sum()})
balance = pd.DataFrame(rows)
balance.to_csv(AUDIT_DIR / "class_balance.csv", index=False)
display(balance.set_index(["class", "subset"]).unstack("subset").reindex(SHORT).round(1))

fig, ax = plt.subplots(1, 3, figsize=(17, 3.6))
b = balance.pivot(index="class", columns="subset", values="share %").reindex(SHORT)
b.plot.bar(ax=ax[0], rot=0, color=["#4c72b0", "#dd8452"])
ax[0].set(title="share of instances (%)", xlabel="")
balance.pivot(index="class", columns="subset", values="images").reindex(SHORT).plot.bar(ax=ax[1], rot=0, color=["#4c72b0", "#dd8452"])
ax[1].set(title="images containing the class", xlabel="", yscale="log")
bins = np.r_[0, np.geomspace(1, 500, 25)]
for s, c in zip(RECORDS, ["#4c72b0", "#dd8452"]):
    ax[2].hist(images.n_objects[images.subset == s], bins=bins, density=True, histtype="step", lw=2, color=c, label=s)
ax[2].set(xscale="symlog", title="objects per image (density)", xlabel="objects")
ax[2].legend()
plt.tight_layout()
plt.show()

density = pd.crosstab(images.subset, pd.cut(images.n_objects, [-1, 0, 9, 49, np.inf], labels=["0", "1-9", "10-49", ">=50"]))
display(density)

### 1.2 Object size and shape, by class and by subset
Size is `sqrt(box area)` in source pixels. The buckets are those of the evaluation (`evaluation.size_bins`): xs < 16 ≤ s < 32 ≤ m.

In [ ]:
def size_summary(df):
    return pd.Series({"objects": len(df), "w median": df.w.median(), "h median": df.h.median(),
                      "size p5": df.sqrt_area.quantile(0.05), "size p50": df.sqrt_area.median(),
                      "size p95": df.sqrt_area.quantile(0.95), "xs %": 100 * (df.sqrt_area < size_bins[0]).mean(),
                      "m %": 100 * (df.sqrt_area >= size_bins[1]).mean(), "max side": df[["w", "h"]].max().max(),
                      "aspect p50": df.aspect.median(), "aspect p90": df.aspect.quantile(0.9)})


size_by_class = objects.groupby(["short", "subset"], observed=True).apply(size_summary).unstack("subset")
size_by_subset = objects.groupby("subset").apply(size_summary)
size_by_class.to_csv(AUDIT_DIR / "size_by_class.csv")
size_by_subset.to_csv(AUDIT_DIR / "size_by_subset.csv")
display(size_by_subset.round(1))
display(size_by_class.round(1))

fig, ax = plt.subplots(1, 3, figsize=(18, 3.8))
bins = np.geomspace(3, 170, 40)
for s, c in zip(RECORDS, ["#4c72b0", "#dd8452"]):
    ax[0].hist(objects.sqrt_area[objects.subset == s], bins=bins, density=True, histtype="step", lw=2, color=c, label=s)
ax[0].set(xscale="log", xlabel="sqrt(box area) [px]", title="object size by subset")
ax[0].legend()
train_obj = objects[objects.subset == "train"]
for k, n in enumerate(SHORT):
    ax[1].hist(train_obj.sqrt_area[train_obj.cls == k], bins=bins, density=True, histtype="step", lw=2, color=COLORS[k], label=n)
    ax[2].hist(train_obj.aspect[train_obj.cls == k], bins=np.linspace(1, 6, 41), density=True, histtype="step", lw=2,
               color=COLORS[k], label=n)
ax[1].set(xscale="log", xlabel="sqrt(box area) [px]", title="object size by class (train)")
ax[2].set(xlabel="long side / short side", title="aspect ratio by class (train)")
for a in ax[:2]:
    for edge in size_bins:
        a.axvline(edge, ls="--", c="gray", lw=1)
ax[1].legend()
plt.tight_layout()
plt.show()

sample = train_obj.sample(min(3000, len(train_obj)), random_state=SAMPLE_SEED)
plt.figure(figsize=(5.5, 5))
plt.scatter(sample.w, sample.h, s=3, alpha=0.4, c=[COLORS[k] for k in sample.cls])
plt.plot([1, 170], [1, 170], "k--", lw=0.8)
plt.xscale("log"); plt.yscale("log"); plt.xlabel("width [px]"); plt.ylabel("height [px]")
plt.title("box width vs height (train sample, colour = class)")
plt.show()

### 1.3 Label quality
* **degenerate:** shorter side < 4 px
* **probable duplicate:** another box of the same class with IoU > 0.7
* **class conflict:** a box of another class with IoU > 0.5

In [ ]:
flags = objects.assign(degenerate=np.minimum(objects.w, objects.h) < 4, duplicate=objects.max_iou_same > 0.7,
                       conflict=objects.max_iou_other > 0.5)
quality = flags.groupby("subset")[["degenerate", "duplicate", "conflict"]].sum()
quality.insert(0, "objects", flags.groupby("subset").size())
quality.to_csv(AUDIT_DIR / "label_quality.csv")
display(quality)

rng = np.random.default_rng(SAMPLE_SEED)
for flag in ("degenerate", "duplicate", "conflict"):
    sel = flags[flags[flag]]
    sel = sel.iloc[rng.permutation(len(sel))[:16]]
    crop_grid([(PATHS[(t.subset, t.image_id)], [t.x1, t.y1, t.x2, t.y2], CLASS_COLORS[t.cls], f"{t.subset} {t.short} {t.w:.0f}x{t.h:.0f}")
               for t in sel.itertuples()], f"{flag} labels (sample)")

### 1.4 Samples
The densest val scene with all its labels, then native-resolution examples of every class (random, `SAMPLE_SEED`).

In [ ]:
rec = max(RECORDS["val"], key=lambda r: len(ANNS["val"][r.image_id]))
a = ANNS["val"][rec.image_id]
img, scale = downscale(rgb(rec.image_path), 2400)
legend = "  ".join(f"{n}={c}" for n, c in zip(SHORT, ["red", "blue", "green", "orange", "purple"]))
show(draw_boxes(img, a.boxes * scale, [CLASS_COLORS[c] for c in a.cls], 2), f"val {rec.image_id}: {len(a)} objects | {legend}", 16)

for k, n in enumerate(SHORT):
    pool = train_obj[train_obj.cls == k]
    few = pool.image_id.drop_duplicates().sample(min(4, pool.image_id.nunique()), random_state=SAMPLE_SEED)   # few decodes
    sel = pool[pool.image_id.isin(few)]
    sel = sel.sample(16, random_state=SAMPLE_SEED, replace=len(sel) < 16)
    crop_grid([(PATHS[("train", t.image_id)], [t.x1, t.y1, t.x2, t.y2], CLASS_COLORS[k], f"{t.sqrt_area:.0f}px")
               for t in sel.itertuples()], f"{n} (train, upscaled for display)", cols=16, px=96)

### Audit findings and what follows from them
Numbers from the tables above.

| Finding | Evidence | Consequence |
|---|---|---|
| **Val is a denser regime than train** | objects/image median 76.5 in val vs 7 in train; 12 of 22 val images have ≥ 50 objects vs 34 of 443 train images | full-image inference must not truncate detections (`max_det_image` 2000); the dev split is stratified on dense images |
| **Strong class imbalance** | train shares: Cargo 49.5%, Box 31.1%, Flatbed 8.7%, Tractor 8.2%, Liquid 2.5% (val similar) | mAP50 is an *unweighted* class mean, so the rare classes weigh as much as Cargo |
| **Liquid AP is the noisiest number** | 20 val instances in 8 images, so one object moves Liquid AP by ~5 points (~1 mAP point) | bootstrap CIs on every reported mAP; dev floor of 25 Liquid instances |
| **Tractor has low effective diversity** | one train image holds 27% of its instances, the top 10 hold 60% | a weak Tractor AP may reflect too few independent scenes, not too few boxes; to be tested in Stage 2 |
| **Val counts are concentrated too** | the 10 richest val images hold > 90% of every class | val AP depends heavily on a handful of scenes; image-level bootstrap |
| **Objects are tiny** | median size 19 px (Cargo, Tractor) to 32 px (Box); 22.6% of train objects < 16 px (Cargo 33%) | train and infer at native resolution: resizing a ~3200 px image to 1024 px would shrink the median truck to ~6 px. Hence 1024 px tiles, `imgsz` = tile size |
| **Largest object side is 161 px** | max of w and h over train and val | tile overlap 200 px > 161 px, so every object lies whole in at least one tile (checked in Stage 3) |
| **Image sizes vary widely** | train 2576–5119 px wide, val 1369–7204 px | sliding-window inference on the full image; the grid snaps the last tile to the border (no padding) |
| **Boxes are axis-aligned boxes of rotated trucks** | aspect p50 1.6–1.9, p90 up to 3.7 (Box, Flatbed) | no canonical "up" in overhead imagery, so vertical flips are valid augmentation (`flipud` 0.5) |
| **Val objects are slightly larger** | median 25.5 px vs 22.0 px; xs share 17% vs 23% | minor shift; size-bucket AP is reported to separate it from model quality |
| **Labels are clean enough** | 10 train / 26 val degenerate boxes, 14 probable train duplicates, 8 train class conflicts, none in val | used as delivered; val labels are never modified |

## Stage 2 · Train/dev split (image level)

Val has 22 images, so it cannot steer any decision without being overfitted. Every choice (epochs, thresholds, ablations) is made on **dev**, a held-out part of train. Val is only reported.

* **Image level.** Tiles of the same scene are strongly correlated, so a whole image goes to either train_core or dev.
* **Method** (`auric.data.splits`). A seeded random search over subsets of round(15% × 443) images keeps candidates that meet per-class instance floors (rare classes need enough dev objects for a usable AP). Among those it picks the one closest to 15% on per-class instance share, per-class image share and dense-image share.
* The split file ships with the code (`configs/splits/`), so every machine uses the identical split. `ensure_split` loads it and checks its parameters and that it partitions train. It creates the file only if it does not exist.

In [ ]:
split, created = ensure_split(SPLIT_FILE, ANNS["train"], NAMES, **cfg["split"])
print(f"{'created' if created else 'loaded'} {SPLIT_FILE}  (search score {split['score']})")

dev_ids = set(split["dev"])
for df in (images, objects):
    df["part"] = np.where(df.subset == "val", "val", np.where(df.image_id.isin(dev_ids), "dev", "train_core"))
PARTS = ["train_core", "dev", "val"]
dense = cfg["split"]["dense_min_objects"]

overview = images.groupby("part").agg(images=("image_id", "size"), objects=("n_objects", "sum"),
                                      dense_images=("n_objects", lambda x: int((x >= dense).sum())),
                                      obj_per_img_mean=("n_objects", "mean")).reindex(PARTS)
train_parts = overview.loc[["train_core", "dev"]]
overview["% of train images"] = 100 * overview.images / train_parts.images.sum()
overview["% of train objects"] = 100 * overview.objects / train_parts.objects.sum()
overview.loc["val", ["% of train images", "% of train objects"]] = np.nan
display(overview.round(1))

inst = pd.crosstab(objects.short, objects.part).reindex(index=SHORT, columns=PARTS)
inst["dev % of train"] = 100 * inst.dev / (inst.dev + inst.train_core)
floors = cfg["split"]["min_dev_instances"]
inst["dev floor"] = [floors.get(n, 0) for n in NAMES]
inst["dev images"] = [int((images[images.part == "dev"][f"n {n}"] > 0).sum()) for n in NAMES]
display(inst.round(1))

sizes = objects.groupby("part").agg(size_p50=("sqrt_area", "median"),
                                    xs_pct=("sqrt_area", lambda x: 100 * (x < size_bins[0]).mean())).reindex(PARTS)
display(sizes.round(1))

share = pd.crosstab(objects.short, objects.part, normalize="columns").reindex(index=SHORT, columns=PARTS)
(100 * share).T.plot.barh(stacked=True, figsize=(10, 2.4), color=COLORS)
plt.title("class share of instances per part (%)")
plt.legend(bbox_to_anchor=(1, 1))
plt.show()

**Split check.** Dev holds 66 of 443 train images (14.9%). Its per-class instance share lies between 12.3% (Tractor) and 16.2% (Liquid). The floors for Flatbed, Tractor and Liquid are met, and class shares match train_core. **Object size is not balanced** (the split did not stratify on it): the median size is similar (dev 21.6 px vs train_core 22.2 px), but dev has more xs objects (< 16 px: 30.5% vs 21.3%), while val has fewer (16.9%). Dev AP will therefore lean toward the hardest size bucket and should be compared with val per size bucket, not only overall. Dev is also not a proxy for val's density: val is far denser than either, as Stage 1 showed.

## Stage 3 · Input formulation: tile cache (build or reuse) or whole scenes

**Tiles** (every experiment except Y2). The detector is trained on fixed-size crops of the train images at native resolution.

| Parameter | Value | Justification |
|---|---|---|
| `size` | 1024 (B0), 512 (A1) | 1024: native resolution (objects are 19–32 px). 512 is enlarged to the 1024 px network input, a 2x up-sampling; training and inference must use the same factor (`inference.tile`, `imgsz`) |
| `overlap` | 200 | larger than the largest object side (161 px), so every object is whole in at least one tile (checked below) |
| `min_visibility` | 0.5 | a box cut by the tile border is kept (clipped) when ≥ 50% of its area is inside the tile, so the model learns to detect objects that are cut at the border. Smaller slivers are dropped; such an object is whole in a neighbouring tile anyway |
| `negative_frac` | 0.25 (A1: 0.07) | a share of object-free tiles is kept so the detector sees background (roads, buildings, vehicles that are not trucks). A1 has ~8x more object-free tiles, so a lower fraction keeps the same ~30% share of negative tiles. The choice is deterministic per tile id, so it is identical on every machine |
| `max_nodata_frac` | 0.5 | object-free tiles that are mostly black no-data (image borders of orthorectified scenes) carry no information and are discarded |

The cache directory is named after these parameters. `ensure_tile_cache` reuses a cache only if its stored parameters and the fingerprint of the source labels match. A mismatching or incomplete cache raises an error; it is never silently rebuilt.

**Whole scenes** (`formulation.mode: scenes`, Y2). The scenes are the samples; no cache is built and the tile cells below are skipped. Ultralytics letterboxes a scene so that its longer side equals `imgsz` (3008): longer scenes are shrunk, shorter ones enlarged, and the objects with them. Scenes range from 2576 to 5119 px in train and up to 7204 px in val, so the scale differs from scene to scene, and between train and val. The first table is therefore the cost of the formulation in the object scale: what size the trucks have *inside the network*, compared with their native size, which tiles keep unchanged.


In [ ]:
if SCENES:
    imgsz = cfg["train"]["args"]["imgsz"]
    images["scale"] = imgsz / images[["width", "height"]].max(axis=1)         # Ultralytics resizes the longer side to imgsz
    obj = objects.merge(images[["subset", "image_id", "scale"]], on=["subset", "image_id"], how="left")
    obj["size_in_net"] = obj.sqrt_area * obj.scale
    by_part = pd.DataFrame({
        "images": images.groupby("part").size(),
        "scale min": images.groupby("part").scale.min(), "scale p10": images.groupby("part").scale.quantile(0.1),
        "scale p50": images.groupby("part").scale.median(), "scale p90": images.groupby("part").scale.quantile(0.9),
        "object px, native (median)": obj.groupby("part").sqrt_area.median(),
        "object px, in network (median)": obj.groupby("part").size_in_net.median(),
        "xs % native (< 16 px)": 100 * obj.groupby("part").sqrt_area.apply(lambda x: (x < size_bins[0]).mean()),
        "xs % in network": 100 * obj.groupby("part").size_in_net.apply(lambda x: (x < size_bins[0]).mean()),
    }).reindex(PARTS)
    by_part.to_csv(AUDIT_DIR / "scene_scale.csv")
    display(by_part.round(2))

    fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
    for part, c in zip(PARTS, ["#4c72b0", "#55a868", "#dd8452"]):
        ax[0].hist(images.scale[images.part == part], bins=np.linspace(0.3, 1.5, 49), histtype="step", lw=2, color=c, label=part,
                   density=True)
        ax[1].hist(obj.size_in_net[obj.part == part], bins=np.geomspace(2, 170, 40), histtype="step", lw=2, color=c, label=part,
                   density=True)
    ax[0].set(xlabel=f"scale = imgsz {imgsz} / longer side", title="scale at which a scene enters the network")
    ax[1].set(xscale="log", xlabel="sqrt(box area) in the network [px]", title="object size seen by the network")
    for edge in size_bins:
        ax[1].axvline(edge, ls="--", c="gray", lw=1)
    ax[0].legend()
    plt.tight_layout()
    plt.show()
else:
    print("tiles formulation: see the tile cache below")

In [ ]:
if not SCENES:
    max_side = objects.loc[objects.subset == "train", ["w", "h"]].max().max()
    print(f"largest train object side {max_side:.0f} px vs overlap {TILING.overlap} px")
    print("train objects not whole in any tile:", count_uncovered(RECORDS["train"], ANNS["train"], TILING.size, TILING.overlap))

    cache = ensure_tile_cache(RECORDS["train"], ANNS["train"], TILE_DIR, TILING, K, WORKERS)
    print(("built" if cache.built else "reused (parameters and source fingerprint match)") + f": {cache.dir}")

In [ ]:
if not SCENES:
    man = cache.manifest.assign(part=lambda m: np.where(m.image_id.isin(dev_ids), "dev", "train_core"))
    STATUS = ["positive", "negative", "neg_sampled_out", "neg_nodata"]
    status = pd.crosstab(man.part, man.status, margins=True, margins_name="all").reindex(columns=STATUS + ["all"], fill_value=0)
    display(status)

    kept = man[man.status.isin(["positive", "negative"])]
    n_source = objects[objects.subset == "train"].groupby("part").size()
    tiles_report = pd.DataFrame({
        "kept tiles": kept.groupby("part").size(),
        "negative share %": 100 * kept.groupby("part").status.apply(lambda s: (s == "negative").mean()),
        "source objects": n_source,
        "object instances in tiles": kept.groupby("part").n_objects.sum(),
        "objects / positive tile": kept[kept.status == "positive"].groupby("part").n_objects.mean(),
        "dropped partials (kept tiles)": kept.groupby("part").n_dropped.sum(),
        "kept negatives with dropped partials": kept[kept.status == "negative"].groupby("part").n_dropped.apply(lambda x: int((x > 0).sum())),
        "positive tiles with nodata > max": kept[kept.status == "positive"].groupby("part").nodata.apply(lambda x: int((x > TILING.max_nodata_frac).sum())),
    }).reindex(["train_core", "dev"])
    tiles_report["instances per source object"] = tiles_report["object instances in tiles"] / tiles_report["source objects"]
    tiles_report.to_csv(AUDIT_DIR / "tiles_report.csv")
    display(tiles_report.round(2))

    fig, ax = plt.subplots(1, 2, figsize=(14, 3.4))
    ax[0].hist([man.nodata[man.status == "positive"], man.nodata[man.status != "positive"]], bins=np.linspace(0, 1, 21),
               label=["positive", "object-free"], log=True)
    ax[0].axvline(TILING.max_nodata_frac, ls="--", c="gray")
    ax[0].set(title="black no-data share per grid tile", xlabel="no-data share")
    ax[0].legend()
    ax[1].hist(kept.n_objects[kept.status == "positive"], bins=np.geomspace(1, kept.n_objects.max() + 1, 30))
    ax[1].set(xscale="log", title="objects per positive tile", xlabel="objects")
    plt.tight_layout()
    plt.show()

**Read-back check.** Every kept tile's label file is parsed with the same parser as the source labels and compared with the source boxes cropped to the tile window. The samples below are drawn from the cached image and label files, which is exactly what Ultralytics will load.

In [ ]:
if not SCENES:
    check = verify_tile_labels(cache, ANNS["train"], K)
    print(f"{len(check)} kept tiles checked | object counts match: {(check.n_file == check.n_expected).all()} | "
          f"classes match: {check.classes_match.all()} | max corner error {check.max_err_px.max():.1e} px | "
          f"all images present: {check.image_exists.all()}")
    assert check.classes_match.all() and check.image_exists.all() and check.max_err_px.max() < 0.01

    pos = kept[kept.status == "positive"].sample(8, random_state=SAMPLE_SEED)
    neg = kept[kept.status == "negative"].sample(4, random_state=SAMPLE_SEED)
    fig, axes = plt.subplots(2, 6, figsize=(24, 8.5))
    for ax, t in zip(axes.flat, pd.concat([pos, neg]).itertuples()):
        img, ann = read_tile(cache, t.tile_id, K)
        ax.imshow(draw_boxes(img[:, :, ::-1], ann.boxes, [CLASS_COLORS[c] for c in ann.cls], 2))
        ax.set_title(f"{t.tile_id} [{t.part}] {t.status}: {len(ann)} obj", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

**Tiling notes** (tiles formulation)
* No object is lost. Every train object lies whole in at least one tile, so dropped partial boxes are slivers of objects that are labelled whole in a neighbouring tile.
* Partial boxes with ≥ 50% visibility are kept, clipped at the tile border. The detector will also see objects cut by tile borders at inference, and stitching merges those duplicates.
* A kept negative tile can contain a dropped sliver (< 50% of an object) without a label. The count is in the table above; it is small and acts like the visibility threshold itself.
* The no-data filter applies only to object-free tiles. Positive tiles are always kept, whatever their black share.

## Stage 4 · YOLO dataset

The run directory is created here. Its id is `<experiment.name>_s<seed>`, so the name is explicit in the config. If the directory already exists, `IF_RUN_EXISTS` decides what happens: `error` stops, `resume` continues (only if the stored config matches), and `overwrite` deletes the run and starts over. The run gets its own image lists and `data.yaml`: tiles are listed by absolute path into the shared cache, scenes by the path of the source image (Ultralytics finds the label file next to it). Ultralytics validates on the dev tiles (or dev scenes) for monitoring only.


### 4.0 Training images (the `subset` section; absent = all train_core images)

Assignment 5.4 asks for the smallest training subset that keeps the full-data mAP50. A config with a `subset` section trains on a *subset of the
train_core images*; dev (monitoring) and val (reporting) are untouched, and the tile cache is the same as for the full run (tiles are listed per image).
The size of a subset is its number of labelled **instances** (`budget_frac` x the train_core instances); images are what is kept or dropped. Both methods
fill the same budget with the same rule (`auric/data/subsets.py`), so a method and its random control cost the same number of labels:

* `random`: images in a seeded random order (`subset.seed`).
* `coverage`: greedy, declared before any subset run: every (class x size bucket) cell gets a target of `budget_frac` of its instances (raised to at most `min_cell` = 20 so the
  rare cells are not left empty); the next image is the one that fills the most missing cell mass per labelled instance it costs; images that fill nothing are added in random order.

The selection is deterministic given the config, so it is reproduced, not stored as a list to trust; the ids are also written to `record.json`.


In [ ]:
SUBSET = cfg.get("subset")
if SUBSET is None:
    TRAIN_IDS, selection = split["train_core"], None
    print(f"training images: all {len(TRAIN_IDS)} train_core images (no subset)")
else:
    selection = select_subset(ANNS["train"], split["train_core"], NAMES, size_bins, **SUBSET)
    TRAIN_IDS = selection["ids"]
    print(f"subset {SUBSET}: {len(TRAIN_IDS)} of {len(split['train_core'])} train_core images, {selection['instances']} instances "
          f"(budget {selection['budget']}); {selection['images_from_coverage']} images picked for coverage, the rest random fill")
    kept = describe_subset(ANNS["train"], split["train_core"], TRAIN_IDS, NAMES, size_bins)
    for name, table in kept.items():
        print(name)
        display(table.rename(index=dict(zip(NAMES, SHORT))) if name == "classes" else table)


In [ ]:
run = create_run(RUN_DIR, cfg, if_exists=IF_RUN_EXISTS)
if SCENES:
    yolo_ds = write_yolo_scene_dataset(run.data_dir, RECORDS["train"], ANNS["train"], TRAIN_IDS, split["dev"], NAMES)
    source = {"formulation": "scenes"}
else:
    yolo_ds = write_yolo_dataset(run.data_dir, cache, TRAIN_IDS, split["dev"], NAMES)
    source = {"formulation": "tiles", "tile_cache": str(cache.dir), "tiling_key": TILING.key}
log_stage(run, "dataset", {**source, "split_file": str(SPLIT_FILE), "split_sha256": sha256(SPLIT_FILE), "subset": selection,
                           "lists": yolo_ds.summary.to_dict("records")})
print(run.dir)
display(yolo_ds.summary.set_index("list"))
print(yolo_ds.data_yaml.read_text())


### 4.1 Class balance as the detector will see it

Two levers act on the class imbalance (configs `Y3`-`Y6`). Both are computed from the label counts of the samples in *this run's train list* (tiles, or scenes):

* `train.args.cls_pw` multiplies the classification loss of class c by `(1 / n_c) ** cls_pw`, scaled to mean 1. It acts on the loss, so it does not depend on which classes share a tile.
* `train.sampler` changes how often each sample is drawn (an epoch keeps its length): object-free samples keep their share; the others are drawn class first (class c with probability proportional to `n_c ** alpha`), then uniformly among the samples that contain it. alpha 1 draws classes as often as they occur but spreads each class's draws evenly over its samples, so it already differs from the plain shuffle; alpha 0 draws every class equally often.

The tables are the *expected* content of one epoch. The thing to read is how little the instance shares move compared with the class draw probabilities: a tile that contains a rare class also contains Cargo and Box, so oversampling it brings those along. "draws per containing sample" is the exposure factor of the samples that hold the class (plain shuffle: 1). With whole scenes (Y2) nearly every sample contains every class, so a sampler has almost nothing to act on there; the class weights do not have that limit.

In [ ]:
if SCENES:
    SAMPLE_COUNTS = np.stack([np.bincount(ANNS["train"][i].cls, minlength=K) for i in TRAIN_IDS])
else:
    train_tiles = cache.kept[cache.kept.image_id.isin(set(TRAIN_IDS))]
    SAMPLE_COUNTS = train_tiles[[f"n_cls{k}" for k in range(K)]].to_numpy()
sampler_cfg, cls_pw = cfg["train"].get("sampler"), cfg["train"]["args"].get("cls_pw", 0.0)
print(f"{len(SAMPLE_COUNTS)} training samples ({int((SAMPLE_COUNTS.sum(1) == 0).sum())} object-free), "
      f"instances per class {SAMPLE_COUNTS.sum(0).tolist()}")

ALPHAS = {"plain shuffle": None, "alpha 1.0": 1.0, "alpha 0.5": 0.5, "alpha 0.0": 0.0}
if sampler_cfg and sampler_cfg["alpha"] not in ALPHAS.values():
    ALPHAS[f"alpha {sampler_cfg['alpha']}"] = sampler_cfg["alpha"]
configured = "plain shuffle" if sampler_cfg is None else f"alpha {sampler_cfg['alpha']}"

tables = {}
for label, alpha in ALPHAS.items():
    probs = np.full(len(SAMPLE_COUNTS), 1 / len(SAMPLE_COUNTS)) if alpha is None else class_aware_probs(SAMPLE_COUNTS, alpha)
    tables[label] = describe_sampling(SAMPLE_COUNTS, probs, SHORT).set_index("class")
share = pd.DataFrame({k: t["share % (expected per epoch)"] for k, t in tables.items()})
exposure = pd.DataFrame({k: t["draws per containing sample"] for k, t in tables.items()})
weights = pd.DataFrame({f"{k}, cls_pw {pw}": class_loss_weights(t["instances (expected per epoch)"], pw)
                        for k, t in tables.items() for pw in (0.5, 1.0)}, index=SHORT)
for name, table in (("class_share_expected", share), ("sample_exposure", exposure), ("class_weights_preview", weights)):
    table.to_csv(AUDIT_DIR / f"{name}_{cfg['experiment']['name']}.csv")
print("expected share of instances per epoch (%):")
display(share.round(1))
print("draws per epoch of each sample that contains the class:")
display(exposure.round(2))
print("class weights if cls_pw were applied to the expected counts of each sampler:")
display(weights.round(2))
print(f"this run: sampler = {configured}, cls_pw = {cls_pw}")

ax = share.T.plot.bar(stacked=True, figsize=(9, 3.2), color=COLORS, rot=0)
ax.axvline(list(ALPHAS).index(configured), c="k", ls=":", lw=1)
ax.set(title="expected share of instances per epoch (%)", ylabel="%")
ax.legend(SHORT, bbox_to_anchor=(1, 1))
plt.show()

## Stage 5 · Training

| | |
|---|---|
| Model | `model.weights`, a COCO-pretrained Ultralytics asset downloaded once to `work_dir/pretrained/`: YOLO11s (B0, D1, A1, T1) or YOLO26s (Y1-Y6) |
| Data | train_core tiles, `imgsz` 1024 = tile size (native resolution; A1: 2x the tile size), or train_core scenes at `imgsz` 3008 (Y2) |
| Recipe | every argument comes from `train.args` in the config: SGD, lr0 0.01, 50 epochs, batch and nbs as set in the config, mosaic (off for the last 10 epochs), HSV, translate/scale, `fliplr` and `flipud` 0.5, AMP, `deterministic`, fixed seed |
| Imbalance | `train.args.cls_pw` (class weights), `train.args.cls` (classification gain), `train.sampler` (class-aware draws). The trainer records the counts, the weights Ultralytics used and the sampler's expected draws in `train/imbalance.json` |
| Checkpoint | `last.pt` after the fixed schedule. No checkpoint is selected on any metric; `best.pt` (dev-tile fitness) is recorded but not evaluated |
| Record | `config.yaml` (resolved config + fingerprint), `record.json` (environment, seed, pretrained and checkpoint hashes, the effective Ultralytics arguments), `train/` (Ultralytics outputs) |

**Reading the YOLO26 curves.** YOLO26 has two heads on the same features, and the training loss is `w * L_one-to-many + (1 - w) * L_one-to-one` with `w` falling linearly from 0.8 to 0.1 over the epochs (Ultralytics, `E2ELoss`). The one-to-one head sees detached features, so the backbone is trained by the one-to-many loss alone, whose weight shrinks as training proceeds. The losses in `results.csv` and in the progress bar are those of the one-to-one head. The dev-tile mAP50 is computed with the one-to-many head and NMS (Ultralytics' default when `nms` is unset), which is also the head used for full-image inference (`inference.head`). `reg_max` is 1, so the third loss is an L1 box loss (`l1_loss`, gain `dfl`), not a DFL.


In [ ]:
pretrained = fetch_pretrained(cfg["model"]["weights"], PRETRAINED_DIR)
seed = cfg["experiment"]["seed"]

if any(e["stage"] == "train_done" for e in read_record(run)):
    print(f"{run.dir} is already trained; reusing {run.weights} (IF_RUN_EXISTS={IF_RUN_EXISTS!r})")
else:
    resume = run.weights.exists()     # an interrupted training, only reachable with IF_RUN_EXISTS="resume"
    log_stage(run, "train_start", {"seed": seed, "resume": resume, "pretrained": str(pretrained),
                                   "pretrained_sha256": sha256(pretrained), "train_args": cfg["train"]["args"],
                                   "sampler": cfg["train"].get("sampler")})
    result = train_yolo(pretrained, yolo_ds.data_yaml, run.train_dir, cfg["train"]["args"], seed, DEVICE, WORKERS,
                        resume=resume, verbose=VERBOSE_TRAIN, sampler=cfg["train"].get("sampler"))
    log_stage(run, "train_done", {
        "seed": seed, "checkpoints": {"last": str(result.last), "best": str(result.best)},
        "last_sha256": sha256(result.last), "best_sha256": sha256(result.best) if result.best else None,
        "results_csv": str(result.results_csv), "effective_args": result.effective_args,
        "imbalance": result.imbalance})

effective = yaml.safe_load((run.train_dir / "args.yaml").read_text())
changed = {k: (v, effective.get(k)) for k, v in cfg["train"]["args"].items() if effective.get(k) != v}
print("config arguments Ultralytics did not use as given (config, effective):", changed or "none")
display(pd.DataFrame(read_record(run))[["stage", "time_utc"]])

In [ ]:
res = pd.read_csv(run.train_dir / "results.csv")
res.columns = res.columns.str.strip()
fig, ax = plt.subplots(1, 3, figsize=(18, 3.8))
loss_keys = [k for k in ("box", "cls", "dfl", "l1") if f"train/{k}_loss" in res]    # YOLO26 (reg_max 1): l1 instead of dfl
for k in loss_keys:
    ax[0].plot(res.epoch, res[f"train/{k}_loss"], label=f"train {k}")
    ax[0].plot(res.epoch, res[f"val/{k}_loss"], "--", label=f"dev {k}")
ax[0].set(title="losses (training samples; YOLO26: one-to-one head)", xlabel="epoch")
ax[0].legend(fontsize=8)
for k in ("precision(B)", "recall(B)", "mAP50(B)", "mAP50-95(B)"):
    ax[1].plot(res.epoch, res[f"metrics/{k}"], label=k)
ax[1].set(title="dev tiles (Ultralytics, monitoring only)", xlabel="epoch")
ax[1].legend(fontsize=8)
ax[2].plot(res.epoch, res["lr/pg0"])
ax[2].set(title="learning rate", xlabel="epoch")
plt.show()

### 5.1 What the imbalance levers did

`imbalance.json` is written by the trainer from the dataset Ultralytics actually loaded (after its own label checks). If a sampler is configured, the expected draws are those of the table in 4.1 for the same samples, and `cls_pw` is applied to the *expected* counts of those draws, so a class is not corrected twice.

In [ ]:
imb_file = run.train_dir / "imbalance.json"
if imb_file.exists():
    imb = json.loads(imb_file.read_text())
    summary = pd.DataFrame({"dataset instances": imb["dataset_counts"],
                            "class weight used": imb["class_weights"] or [1.0] * len(NAMES)}, index=SHORT)
    print(f"cls_pw = {imb['cls_pw']}, sampler = {imb['sampler'] and {k: v for k, v in imb['sampler'].items() if k != 'per_class'}}")
    display(summary.round(2))
    if imb["sampler"]:
        display(pd.DataFrame(imb["sampler"]["per_class"]).set_index("class").set_axis(SHORT, axis=0).round(2))
else:
    print("no imbalance.json: this run was trained before the trainer recorded it (checkpoint imported from an earlier session)")

## Stage 6 · Full-image evaluation (dev, val)

* **Inference.** Each full image is cut into the same tile grid as in training. The tiles are run through the detector in batches; the boxes are shifted back to image coordinates and merged across tiles with class-aware NMS (`inference.*` in the config). For the scenes formulation (Y2) the grid has one window, the whole scene, at the training `imgsz`, and there is nothing to merge. YOLO26 decodes with the head named in `inference.head`; how that head and the NMS variants compare is the subject of notebook 02.
* **Pipeline ceiling.** Ground truth pushed through the same tiling and stitching gives the best AP that tiling and merging allow. The model cannot exceed it.
* **Metrics.** All are computed on full images against the original labels. The primary metric is **mAP@0.5** with COCO 101-point interpolation (pycocotools-identical) and a 95% image-bootstrap CI. Diagnostics use one operating point, the confidence that maximises micro-F1 (`auric/evaluation/diagnostics.py`).
* Predictions and evaluations are overwritten when this stage is re-run. They are deterministic given the weights and the inference settings, and `record.json` stores both.

In [ ]:
run = open_run(RUN_DIR, cfg)
ic, ec = cfg["inference"], cfg["evaluation"]
EVAL_SETS = {"dev": (subset(RECORDS["train"], split["dev"]), ANNS["train"]), "val": (RECORDS["val"], ANNS["val"])}

ceiling = {}
for part in ec["splits"]:
    recs, anns = EVAL_SETS[part]
    gts = {r.image_id: anns[r.image_id] for r in recs}
    oracle = stitch_ground_truth(recs, gts, ic["tile"], ic["overlap"], TILING.min_visibility, ic["merge_iou"],
                                 ic["edge_margin"], ic["max_det_image"])
    ceiling[part] = evaluate(gts, oracle, NAMES, ec["size_bins"], bootstrap=0).summary["mAP50"]
print("pipeline ceiling mAP50:", {k: round(v, 4) for k, v in ceiling.items()})

predictor = TiledPredictor(run.weights, ic, DEVICE)
weights_sha = sha256(run.weights)
reports = {}
for part in ec["splits"]:
    recs, anns = EVAL_SETS[part]
    gts = {r.image_id: anns[r.image_id] for r in recs}
    dets = predictor.predict(recs, desc=f"predict {part}")
    save_detections(dets, run.predictions(part))
    rep = evaluate(gts, dets, NAMES, ec["size_bins"], ec["bootstrap"], ec["seed"])
    rep.summary["settings"] = {"evaluation": ec, "inference": ic, "weights_sha256": weights_sha,
                               "pipeline_ceiling_mAP50": ceiling[part]}
    rep.save(run.eval_dir(part))
    log_stage(run, f"evaluate_{part}", {"weights_sha256": weights_sha, "inference": ic, "evaluation": ec,
                                        "images": [r.image_id for r in recs], "mAP50": rep.summary["mAP50"]})
    reports[part] = rep

In [ ]:
def headline(r):
    s, op = r.summary, r.summary["operating_point"]
    ci = s["mAP50_ci95"] or [np.nan, np.nan]
    row = {"images": s["n_images"], "objects": s["n_gt"], "mAP50": s["mAP50"], "mAP50 CI95": f"[{ci[0]:.3f}, {ci[1]:.3f}]",
           "pipeline ceiling": s["settings"]["pipeline_ceiling_mAP50"], "mAP50-95": s["mAP50_95"], "mAP75": s["mAP75"],
           "AP50 class-agnostic": s["AP50_class_agnostic"], "mAP50 (Ultralytics interp.)": s["mAP50_ultralytics"]}
    row.update({f"AP50 {k.split()[0]}": v for k, v in s["AP50_by_size"].items()})
    row.update({"conf*": op["conf"], "P@conf*": op["precision"], "R@conf*": op["recall"], "F1@conf*": op["f1"]})
    return row


display(pd.DataFrame({p: headline(r) for p, r in reports.items()}))
for p, r in reports.items():
    print(p)
    display(r.per_class.set_index("class"))

In [ ]:
r = reports["val"]
fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
for k, n in enumerate(NAMES):
    c = r.pr_curves[r.pr_curves["class"] == n]
    ax[0].plot(c.recall, c.precision, color=COLORS[k], label=f"{SHORT[k]} {r.per_class.set_index('class').AP50[n]:.3f}")
ax[0].set(xlabel="recall", ylabel="precision", title="val PR curves (IoU 0.5, interpolated)", xlim=(0, 1), ylim=(0, 1.02))
ax[0].legend()
ps = r.per_size.set_index("bucket")[[f"AP50 {n}" for n in NAMES]]
ps.set_axis(SHORT, axis=1).plot.bar(ax=ax[1], color=COLORS, rot=0)
ax[1].set(title="val AP50 per size bucket (sqrt area, px)", ylim=(0, 1))
plt.show()
display(r.per_size.set_index("bucket"))

In [ ]:
OUTCOME_COLORS = {"TP": (60, 200, 60), "FN_class": (255, 140, 0), "FN_loc": (255, 220, 0), "FN_lowscore": (0, 200, 255),
                  "FN_nobox": (255, 0, 255), "FP": (255, 30, 30)}
GT_OUT = ["TP", "FN_class", "FN_loc", "FN_lowscore", "FN_nobox"]

fig, axes = plt.subplots(1, len(reports), figsize=(8 * len(reports), 5.5), squeeze=False)
for ax, (p, r) in zip(axes[0], reports.items()):
    cm = r.confusion
    norm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0)
    ax.imshow(norm.to_numpy(float), cmap="Blues", vmin=0, vmax=1)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm.iat[i, j]), ha="center", va="center", fontsize=8,
                    color="white" if norm.iat[i, j] > 0.5 else "black")
    ax.set_xticks(range(cm.shape[1]), SHORT + ["missed"], rotation=30)
    ax.set_yticks(range(cm.shape[0]), SHORT + ["background"])
    ax.set(xlabel="predicted", ylabel="ground truth", title=f"{p}: class-agnostic matches at conf*")
plt.tight_layout()
plt.show()

for p, r in reports.items():
    print(p, "- ground-truth outcomes / detection outcomes at conf*")
    display(r.fn_breakdown.set_index("class"))
    display(r.fp_breakdown.set_index("class"))
fig, axes = plt.subplots(1, len(reports), figsize=(8 * len(reports), 4), squeeze=False)
for ax, (p, r) in zip(axes[0], reports.items()):
    fn = r.fn_breakdown.set_index("class").drop(index="all")
    fn[GT_OUT].div(fn.n_gt, axis=0).set_axis(SHORT, axis=0).plot.barh(
        stacked=True, ax=ax, color=[np.array(OUTCOME_COLORS[k]) / 255 for k in GT_OUT])
    ax.set(title=f"{p}: fate of ground-truth objects at conf*", xlim=(0, 1))
plt.show()

In [ ]:
r = reports["val"]
pi = r.per_image
conf = r.summary["operating_point"]["conf"]
worst = pi[pi.n_gt >= 20].sort_values("recall").image_id.iloc[0]
display(pi.sort_values("recall").head(8))

g = r.gt_outcomes[r.gt_outcomes.image_id == worst]
d = r.det_outcomes[(r.det_outcomes.image_id == worst) & (r.det_outcomes.outcome != "TP")]
img = rgb(PATHS[("val", worst)])
img = draw_boxes(img, d[["x1", "y1", "x2", "y2"]].to_numpy(), [OUTCOME_COLORS["FP"]] * len(d), 2)
img = draw_boxes(img, g[["x1", "y1", "x2", "y2"]].to_numpy(), [OUTCOME_COLORS[o] for o in g.outcome], 2)
show(downscale(img, 2400)[0], f"val {worst} (lowest recall among images with >= 20 objects): TP green | FN class orange, "
     f"loc yellow, low-score cyan, no-box magenta | FP red (conf* = {conf:.3f})", 16)


def gt_items(outcome, n=32):
    sel = r.gt_outcomes[r.gt_outcomes.outcome == outcome].sample(frac=1, random_state=SAMPLE_SEED).head(n)
    return [(PATHS[("val", t.image_id)], [t.x1, t.y1, t.x2, t.y2], OUTCOME_COLORS[outcome],
             f"{SHORT[t.cls]}" + (f"->{SHORT[t.confused_as]}" if t.confused_as >= 0 else "") + f" {t.sqrt_area:.0f}px")
            for t in sel.itertuples()]


def det_items(outcome, n=32):
    sel = r.det_outcomes[r.det_outcomes.outcome == outcome].sort_values("score", ascending=False).head(n)
    return [(PATHS[("val", t.image_id)], [t.x1, t.y1, t.x2, t.y2], OUTCOME_COLORS["FP"], f"{SHORT[t.cls]} {t.score:.2f}")
            for t in sel.itertuples()]


crop_grid(gt_items("FN_nobox"), "val: missed, no box at any score (FN_nobox)")
crop_grid(gt_items("FN_lowscore"), "val: found only below conf* (FN_lowscore)")
crop_grid(gt_items("FN_class"), "val: localised but wrong class (FN_class: true->predicted)")
crop_grid(det_items("bkg"), "val: highest-scoring background false positives")
crop_grid(det_items("cls"), "val: highest-scoring wrong-class detections")

## Observations (fill in after the run)

* **Headline:** dev and val mAP50 with CI; gap to the pipeline ceiling.
* **Where AP is lost:** misses vs class confusion vs localisation vs false positives, by class and by size.
* **Hypotheses for Stage 2**, each tied to an error population above and to an audit finding (imbalance, concentration, size).
* **Imbalance:** per-class dev AP and recall against the control (Y1) for the class weights, the sampler and both; the realised instance shares in 5.1.
* **Formulation:** tiles vs whole scenes (Y1 vs Y2), by size bucket and by scene size: does the shrinkage of the large scenes show in the AP of the small objects.
